# Company Segmentation

This notebook documents the business-driven segmentation used in the
Campania Financial & ESG Forecasting project.

The original company-level dataset is proprietary and cannot be redistributed.
For this public portfolio, the methodology is demonstrated on a small
synthetic dataset that reproduces the structure required by the segmentation
pipeline.

## Segmentation dimensions

Companies are organized according to:

- company size;
- economic sector and macro-sector from ATECO codes;
- operational province;
- ESG rating;
- employment-growth dynamics.

These groups are used to compare model performance and business outcomes
across heterogeneous company profiles.

In [ ]:
import numpy as np
import pandas as pd

## 1. Synthetic demonstration data

The synthetic records below do not represent real companies.
They are included only to make the segmentation logic reproducible.

In [ ]:
df = pd.DataFrame(
    {
        "company_id": [f"COMP_{i:03d}" for i in range(1, 13)],
        "employees_2021": [4, 18, 75, 9, 32, 120, 6, 44, 58, 15, 82, 27],
        "employees_2024": [5, 23, 88, 7, 30, 156, 10, 38, 62, 21, 70, 36],
        "ateco_code": [
            "25.62", "47.11", "62.01", "41.20",
            "49.41", "64.19", "56.10", "71.12",
            "35.11", "86.21", "68.20", "46.90"
        ],
        "province": [
            "Napoli", "Salerno", "Caserta", "Avellino",
            "Benevento", "Napoli", "Salerno", "Caserta",
            "Avellino", "Benevento", "Napoli", "Salerno"
        ],
        "esg_score_2021": [
            0.18, 0.41, 0.81, 0.29,
            0.63, 0.76, 0.52, 0.71,
            0.34, 0.88, 0.57, 0.24
        ],
    }
)

df

## 2. Company size

The original project uses three employee-based size groups:

- **Micro:** 0–9 employees
- **Small:** 10–49 employees
- **Medium/Large:** 50+ employees

The grouping is deterministic and is used for model-performance comparisons.

In [ ]:
def classify_company_size(employees):
    if pd.isna(employees):
        return np.nan
    if employees <= 9:
        return "micro"
    if employees <= 49:
        return "small"
    return "medium/large"


df["company_size"] = df["employees_2024"].apply(classify_company_size)

df[["company_id", "employees_2024", "company_size"]]

## 3. ATECO sector mapping

The first two digits of the Italian ATECO code identify the economic division.
The project first maps these divisions into interpretable sectors and then
aggregates them into five macro-sectors used in the downstream analysis.

In [ ]:
def extract_ateco_division(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    digits = "".join(ch for ch in text if ch.isdigit())

    return digits[:2] if len(digits) >= 2 else np.nan


sector_map = {
    "Manufacturing": [f"{i:02d}" for i in range(10, 34)],
    "Energy": ["35"],
    "Water_Waste": ["36", "37", "38", "39"],
    "Construction": ["41", "42", "43"],
    "Trade": ["45", "46", "47"],
    "Transport": ["49", "50", "51", "52", "53"],
    "Accommodation_Food": ["55", "56"],
    "Information_Communication": ["58", "59", "60", "61", "62", "63"],
    "Finance": ["64", "65", "66"],
    "Real_Estate": ["68"],
    "Professional_Technical": ["69", "70", "71", "72", "73", "74", "75"],
    "Business_Support": ["77", "78", "79", "80", "81", "82"],
    "Education": ["85"],
    "Health": ["86", "87", "88"],
    "Arts_Sports": ["90", "91", "92", "93"],
    "Other_Services": ["94", "95", "96"],
}

division_to_sector = {
    division: sector
    for sector, divisions in sector_map.items()
    for division in divisions
}

macro_sector_map = {
    "Manufacturing": "Industry and Construction",
    "Construction": "Industry and Construction",

    "Energy": "Networks, Logistics and Infrastructure",
    "Water_Waste": "Networks, Logistics and Infrastructure",
    "Transport": "Networks, Logistics and Infrastructure",

    "Trade": "Trade and Consumer Services",
    "Accommodation_Food": "Trade and Consumer Services",

    "Information_Communication": "Knowledge Economy and Advanced Services",
    "Professional_Technical": "Knowledge Economy and Advanced Services",
    "Business_Support": "Knowledge Economy and Advanced Services",

    "Finance": "Finance, Real Estate and Collective Services",
    "Real_Estate": "Finance, Real Estate and Collective Services",
    "Education": "Finance, Real Estate and Collective Services",
    "Health": "Finance, Real Estate and Collective Services",
    "Arts_Sports": "Finance, Real Estate and Collective Services",
    "Other_Services": "Finance, Real Estate and Collective Services",
}

df["ateco_division"] = df["ateco_code"].apply(extract_ateco_division)
df["sector"] = df["ateco_division"].map(division_to_sector)
df["macro_sector"] = df["sector"].map(macro_sector_map)

df[["company_id", "ateco_code", "sector", "macro_sector"]]

## 4. Operational province

Geographic segmentation is based on the five provinces of Campania:

- Avellino
- Benevento
- Caserta
- Napoli
- Salerno

In the original project, missing province information was recovered through
a separate GenAI-assisted web-search workflow documented in Notebook 06.

In [ ]:
campania_provinces = {
    "Avellino",
    "Benevento",
    "Caserta",
    "Napoli",
    "Salerno",
}

df["province_valid"] = df["province"].isin(campania_provinces)

df[["company_id", "province", "province_valid"]]

## 5. ESG rating

For descriptive segmentation, the normalized 2021 ESG score is divided into
four intervals:

- **D:** 0.00–0.25
- **C:** 0.25–0.50
- **B:** 0.50–0.75
- **A:** 0.75–1.00

This rating is a project-specific analytical grouping and should not be
interpreted as an external credit or sustainability rating.

In [ ]:
df["esg_rating"] = pd.cut(
    df["esg_score_2021"],
    bins=[0.00, 0.25, 0.50, 0.75, 1.00],
    labels=["D", "C", "B", "A"],
    include_lowest=True,
)

df[["company_id", "esg_score_2021", "esg_rating"]]

## 6. Employment-growth dynamics

Employment dynamics are evaluated over a three-year horizon using the
compound annual growth rate (CAGR).

The classification follows the thresholds used in the project:

- **growing:** CAGR > +10%
- **stable:** CAGR between −10% and +10%
- **declining:** CAGR < −10%

In [ ]:
def employment_cagr(start_employees, end_employees, years=3):
    if (
        pd.isna(start_employees)
        or pd.isna(end_employees)
        or start_employees <= 0
        or years <= 0
    ):
        return np.nan

    return (end_employees / start_employees) ** (1 / years) - 1


def classify_employment_growth(cagr):
    if pd.isna(cagr):
        return np.nan
    if cagr > 0.10:
        return "growing"
    if cagr < -0.10:
        return "declining"
    return "stable"


df["employment_cagr"] = df.apply(
    lambda row: employment_cagr(
        row["employees_2021"],
        row["employees_2024"],
        years=3,
    ),
    axis=1,
)

df["employment_growth_class"] = (
    df["employment_cagr"].apply(classify_employment_growth)
)

df[
    [
        "company_id",
        "employees_2021",
        "employees_2024",
        "employment_cagr",
        "employment_growth_class",
    ]
]

## 7. Segmented company profiles

The resulting features can be used to evaluate predictions and business
patterns across comparable company groups.

In [ ]:
segmentation_columns = [
    "company_id",
    "company_size",
    "province",
    "sector",
    "macro_sector",
    "esg_rating",
    "employment_growth_class",
]

segmented_df = df[segmentation_columns].copy()

segmented_df

## 8. Segment distributions

In [ ]:
summary = {
    "company_size": segmented_df["company_size"].value_counts(),
    "macro_sector": segmented_df["macro_sector"].value_counts(),
    "province": segmented_df["province"].value_counts(),
    "esg_rating": segmented_df["esg_rating"].value_counts(),
    "employment_growth_class": segmented_df["employment_growth_class"].value_counts(),
}

for name, counts in summary.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(counts)

## Key Takeaways

This notebook converts heterogeneous company attributes into deterministic,
interpretable business segments.

The segmentation is not an unsupervised clustering algorithm. Instead, it
uses explicit business and statistical rules so that downstream model
performance can be compared transparently across:

- firm size;
- geography;
- economic activity;
- ESG profile;
- employment dynamics.

## Next Step

The next notebook benchmarks the forecasting models used in the project:
Linear Regression, Random Forest, XGBoost, Prophet and LSTM.